# LEVIR-CD · Kontrollü kıyas ve BIT

Colab'da **Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU** seç. Kontrollü kıyas için **1–3** ile Drive'ı ve veriyi hazırla, sonra doğrudan **8** numaralı hücreyi çalıştır. **4–7** numaralı hücreler BIT deneyine ait.

Arşivler [resmî LEVIR-CD kaynağından](https://justchenhao.github.io/LEVIR/) gelir. Eğitim kayıtları ve sonuçlar Drive'a yazılır.

## 1. GPU ve proje kodu
GPU görünmüyorsa hücre burada durur; veri indirmez.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/apeates/change-detection.git'
REPO_DIR = Path('/content/change-detection')
DATA_ROOT = Path('/content/levir_cd_data')
BATCH_SIZE = 2
SEED = 42
FULL_EPOCHS = 30
PROJECT_REF = 'main'  # Eski bir koşuya dönerken koşunun Git SHA değerini yaz.

if not shutil.which('nvidia-smi') or subprocess.run(['nvidia-smi', '-L'], check=False).returncode != 0:
    raise RuntimeError('Colab çalışma zamanı ayarından GPU seçip bu hücreyi tekrar çalıştır.')

clone_needed = not REPO_DIR.exists()
if clone_needed:
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    '--branch', 'main', REPO_URL, str(REPO_DIR)], check=True)
if not clone_needed or PROJECT_REF != 'main':
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', PROJECT_REF], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print(f'Git commit: {REVISION}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO_DIR / 'requirements-train.txt'), 'gdown>=5.2,<7'], check=True)
subprocess.run([sys.executable, '-c',
                "import torch; assert torch.cuda.is_available(), 'PyTorch GPU erişimi yok'; print(torch.cuda.get_device_name(0))"],
               check=True)
sys.path.insert(0, str(REPO_DIR))

## 2. Eğitim kayıtları için Drive'ı bağla
Colab izin isterse kendi Google hesabında onayla.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
RUNS_ROOT = Path('/content/drive/MyDrive/change-detection-runs')
RUN_ROOT = RUNS_ROOT / 'bit_levir_e30'
DRIVE_ARCHIVES = None  # Örnek: Path('/content/drive/MyDrive/levir-cd')
print(f'Eğitim kayıtları {RUN_ROOT}')

## 3. LEVIR-CD verisini hazırla

Kontrollü kıyas için önce `PREPARE_TRAIN = False` ve `PREPARE_TEST = False` ile 64 doğrulama çiftini hazırla. Sonra `PREPARE_TEST = True` ile aynı hücreyi tekrar çalıştırıp 128 test çiftini hazırla. BIT eğitimi için `PREPARE_TRAIN = True` ve `PREPARE_TEST = False` ile 445 eğitim ve 64 doğrulama çifti hazırlanır.

In [ ]:
import tempfile
import zipfile
from pathlib import PurePosixPath

import gdown
from levir_data import list_samples

PREPARE_TEST = False  # Nihai test için True seç.
PREPARE_TRAIN = False  # BIT eğitimi için True seç.
SPLITS = {
    'train': ('18GuoCuBn48oZKAlEo-LrNwABrFhVALU-', 445),
    'val': ('1BqSt4ueO7XAyQ_84mUjswUSJt13ZBuzG', 64),
}
if PREPARE_TEST:
    SPLITS = {'test': ('1jj3qJD_grJlgIhUWO09zibRGJe0R4Tn0', 128)}
ARCHIVE_CACHE = Path('/content/levir_cd_archives')

def verify_split(directory, expected):
    count = len(list_samples(directory))
    if count != expected:
        raise ValueError(f'{directory.name}: {count} eşleşme bulundu, {expected} bekleniyor. ZIP adını kontrol et.')
    return count

def get_archive(split, file_id):
    if DRIVE_ARCHIVES is not None:
        archive = Path(DRIVE_ARCHIVES) / f'{split}.zip'
    else:
        ARCHIVE_CACHE.mkdir(parents=True, exist_ok=True)
        archive = ARCHIVE_CACHE / f'{split}.zip'
        if not zipfile.is_zipfile(archive):
            downloaded = gdown.download(id=file_id, output=str(archive), quiet=False)
            if not downloaded:
                raise RuntimeError(f'{split}.zip indirilemedi. Hata mesajını kontrol et veya DRIVE_ARCHIVES kullan.')
    if not zipfile.is_zipfile(archive):
        raise ValueError(f'Geçerli ZIP bulunamadı: {archive}')
    return archive

def extract_split(archive, destination, expected):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=destination.parent) as directory:
        staged = Path(directory) / destination.name
        for folder in ('A', 'B', 'label'):
            (staged / folder).mkdir(parents=True)
        with zipfile.ZipFile(archive) as source:
            for entry in source.infolist():
                member = PurePosixPath(entry.filename)
                folder = member.parent.name
                if entry.is_dir() or member.suffix.lower() != '.png' or member.name.startswith('.'):
                    continue
                if folder not in ('A', 'B', 'label'):
                    continue
                target = staged / folder / member.name
                with source.open(entry) as incoming, target.open('xb') as outgoing:
                    shutil.copyfileobj(incoming, outgoing)
        verify_split(staged, expected)
        staged.replace(destination)

for split, (file_id, expected) in SPLITS.items():
    if split == 'train' and not PREPARE_TRAIN:
        continue
    destination = DATA_ROOT / split
    if not destination.exists():
        extract_split(get_archive(split, file_id), destination, expected)
    print(f'{split}: {verify_split(destination, expected)} eşleşen görüntü çifti')

## 4. Kısa GPU provası

Dört eğitim parçası ve bir doğrulama çiftiyle iki tur çalışır. Çıktıda güncelleme sayısını ve yeniden yüklenen checkpoint sonucunu kontrol et.

In [ ]:
def run_training(phase, epochs, smoke_tiles=None):
    output = RUN_ROOT / phase
    command = [sys.executable, '-u', 'train_unet.py', '--model', 'bit', '--device', 'cuda',
               '--data-root', str(DATA_ROOT), '--output-dir', str(output),
               '--epochs', str(epochs), '--batch-size', str(BATCH_SIZE), '--seed', str(SEED),
               '--lr-schedule', 'linear', '--revision', REVISION]
    if smoke_tiles is not None:
        command.extend(['--smoke-tiles', str(smoke_tiles)])
    if (output / 'last.pt').is_file():
        command.append('--resume')
    subprocess.run(command, cwd=REPO_DIR, check=True, stderr=subprocess.STDOUT,
                   env={**os.environ, 'MPLBACKEND': 'Agg'})


run_training('smoke', epochs=2, smoke_tiles=4)

## 5. Tam BIT eğitimi

Prova tamamlanınca `RUN_FULL = True` yap. 30 turda 106.800 güncelleme yapılır. Oturum kapanırsa aynı Git sürümü ve veriyle 1–4. adımları çalıştırıp bu hücreyi aç. Drive'daki son tamamlanan turdan devam eder.

In [ ]:
import torch

RUN_FULL = False

if RUN_FULL:
    smoke_checkpoint = RUN_ROOT / 'smoke' / 'last.pt'
    if not smoke_checkpoint.is_file():
        raise RuntimeError('Önce 4. adımda GPU provasını tamamla.')
    smoke = torch.load(smoke_checkpoint, map_location='cpu', weights_only=True)
    if smoke['epoch'] != 2 or smoke['architecture'] != 'bit' or smoke['config']['revision'] != REVISION:
        raise RuntimeError('Bu kod sürümüyle iki turluk BIT provasını tamamla.')
    run_training('full', epochs=FULL_EPOCHS)

## 6. Doğrulama eşiğini seç

Tam eğitim bittikten sonra çalıştır. Doğrulamada seçilen eşik ve checkpoint kimliği Drive'da `validation/selection.json` dosyasına yazılır.

In [ ]:
subprocess.run([sys.executable, '-u', 'evaluate_bit.py', '--run', str(RUN_ROOT),
                '--data-root', str(DATA_ROOT), '--device', 'cuda'],
               cwd=REPO_DIR, check=True, env={**os.environ, 'MPLBACKEND': 'Agg'})

## 7. Nihai LEVIR-CD testi

6. adımı bitir, ardından 3. adımda `PREPARE_TEST = True` seçip çalıştır. Bu hücre, BIT'i kayıtlı Early Fusion ve Siamese modelleriyle ve piksel farkı yöntemiyle aynı test çiftlerinde ölçer.

In [ ]:
import json
from IPython.display import Image, display

final_dir = RUN_ROOT / 'test'
report = final_dir / 'summary.json'
if not report.is_file():
    if not (DATA_ROOT / 'test').is_dir():
        raise RuntimeError('3. hücrede PREPARE_TEST = True seçip çalıştır.')
    subprocess.run([sys.executable, '-u', 'evaluate_test.py', '--data-root', str(DATA_ROOT),
                    '--runs-root', str(RUNS_ROOT), '--output-dir', str(final_dir), '--bit-run', str(RUN_ROOT),
                    '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--revision', REVISION],
                   cwd=REPO_DIR, check=True, env={**os.environ, 'MPLBACKEND': 'Agg'})
summary = json.loads(report.read_text())
method_labels = summary.get('method_labels', {})
for name, subset in summary['subsets'].items():
    label = 'Resmî test' if name == 'full' else 'Örtüşen çiftler hariç'
    print(f"\n{label} · {subset['samples']} çift")
    print(f"{'Yöntem':22} {'Eşik':>5} {'F1':>8} {'IoU':>8} {'Precision':>9} {'Recall':>8} {'FP':>12} {'FN':>12}")
    for method, row in subset['methods'].items():
        threshold = summary['selection'][method]['threshold']
        method_label = method_labels.get(method, method.replace('_', ' '))
        print(f"{method_label:22} {threshold:5g} {row['metrics']['F1']:8.2%} "
              f"{row['metrics']['IoU']:8.2%} {row['metrics']['Precision']:9.2%} {row['metrics']['Recall']:8.2%} "
              f"{row['counts']['fp']:12,} {row['counts']['fn']:12,}")
    delta = subset['bit_vs_siamese_f1_delta'] * 100
    low, high = (value * 100 for value in subset['bit_vs_siamese_f1_delta_ci95'])
    print(f'BIT – Siamese F1 farkı {delta:+.2f} puan, %95 aralık [{low:+.2f}, {high:+.2f}]')
print(f'\nDrive çıktıları: {final_dir}')
for example in summary['examples'].values():
    display(Image(filename=str(final_dir / example['figure']), width=1000))

## Drive çıktıları

`change-detection-runs/bit_levir_e30/` altında `full/history.csv` eğitim geçmişini ve `full/run.json` koşu ayarlarını içerir. BIT deneyi 12 tamamlanmış turda durduruldu. En iyi doğrulama F1'i 1. turda %63,05, 12. turdaki F1'i %34,24.

## 8. LEVIR-CD kontrollü kıyası

1–3. adımlarda GPU'yu, Drive'ı, doğrulama ve test çiftlerini hazırla. Bu hücre Drive'daki Early Fusion ve 30 turluk Siamese checkpoint'lerini kullanır. Eşikler doğrulamada belirlenir. Beş yöntem testte aynı geçerli piksel alanında ölçülür. Çıktılar `change-detection-runs/levir_controlled/` altına kaydedilir.

In [ ]:
from IPython.display import Image, Markdown, display

controlled_dir = RUNS_ROOT / 'levir_controlled'
reports = ('classical_test.csv', 'robustness.csv', 'robustness.png',
           'resources.json', 'protocol.json', 'summary.md')
if not controlled_dir.exists():
    if not all((DATA_ROOT / split).is_dir() for split in ('val', 'test')):
        raise RuntimeError('Önce 3. adımda doğrulama ve test verisini hazırla.')
    subprocess.run([sys.executable, '-u', 'benchmark_levir.py', '--data-root', str(DATA_ROOT),
                    '--runs-root', str(RUNS_ROOT), '--output-dir', str(controlled_dir), '--device', 'cuda'],
                   cwd=REPO_DIR, check=True, stderr=subprocess.STDOUT,
                   env={**os.environ, 'MPLBACKEND': 'Agg'})
missing = [name for name in reports if not (controlled_dir / name).is_file()]
if missing:
    raise RuntimeError(f'{controlled_dir} içinde eksik raporlar: {", ".join(missing)}')
print(f'Drive çıktıları · {controlled_dir}')
display(Markdown((controlled_dir / 'summary.md').read_text(encoding='utf-8')))
display(Image(filename=str(controlled_dir / 'robustness.png')))